In [ ]:
# =============================================
# IMPORTY
# =============================================
import os
import sys
import math

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"✓ Device: {device}")
print(f"✓ PyTorch: {torch.__version__}")

_nb_dir = os.path.abspath('')
if os.path.basename(_nb_dir) in ('experiments', 'examples', 'train'):
    os.chdir(os.path.dirname(_nb_dir))
sys.path.insert(0, os.path.join(os.getcwd(), 'PFNs'))
print(f"✓ Working directory: {os.getcwd()}")

from pfns.model import bar_distribution as bar_dist_module
from pfns.train import train, MainConfig, OptimizerConfig, TransformerConfig, BatchShapeSamplerConfig
from pfns.model.encoders import EncoderConfig
from pfns.model.bar_distribution import BarDistributionConfig
from pfns.priors.prior import AdhocPriorConfig
from pfns.priors.fast_gp import get_batch as get_batch_for_gp

print("✓ PFNs načteny")

In [ ]:
# =============================================
# LOAD_FOR_INFERENCE + PRIOR WRAPPERY
# =============================================

def get_batch_for_gp_random_hps(batch_size, seq_len, num_features,
                                  device='cpu', hyperparameters=None, **kwargs):
    """Trénovací prior: ℓ ~ U(0.05,1.0), σ_f ~ LN(0,0.5), noise ~ 10^U(-3,-1)"""
    random_hps = {
        "lengthscale": float(np.random.uniform(0.05, 1.0)),
        "outputscale": float(np.random.lognormal(0, 0.5)),
        "noise":       float(10 ** np.random.uniform(-3, -1)),
    }
    return get_batch_for_gp(batch_size, seq_len, num_features,
                             device=device, hyperparameters=random_hps, **kwargs)


def get_batch_for_gp_narrow_prior(batch_size, seq_len, num_features,
                                   device='cpu', hyperparameters=None, **kwargs):
    """Úzký prior: ℓ ~ LN(-1, 0.2²) ≈ soustředěno kolem ℓ≈0.37"""
    ell = float(np.random.lognormal(-1.0, 0.2))
    ell = max(0.01, ell)
    random_hps = {
        "lengthscale": ell,
        "outputscale": float(np.random.lognormal(0, 0.5)),
        "noise":       float(10 ** np.random.uniform(-3, -1)),
    }
    return get_batch_for_gp(batch_size, seq_len, num_features,
                             device=device, hyperparameters=random_hps, **kwargs)


def get_batch_for_gp_wide_prior(batch_size, seq_len, num_features,
                                 device='cpu', hyperparameters=None, **kwargs):
    """Široký prior: ℓ ~ LN(-1, 1.5²) ~ ℓ∈[0.01, 10]"""
    ell = float(np.random.lognormal(-1.0, 1.5))
    ell = max(0.01, ell)
    random_hps = {
        "lengthscale": ell,
        "outputscale": float(np.random.lognormal(0, 0.5)),
        "noise":       float(10 ** np.random.uniform(-3, -1)),
    }
    return get_batch_for_gp(batch_size, seq_len, num_features,
                             device=device, hyperparameters=random_hps, **kwargs)


def get_batch_for_gp_bimodal_prior(batch_size, seq_len, num_features,
                                    device='cpu', hyperparameters=None, **kwargs):
    """Bimodální prior: ℓ ~ 0.5·N(0.1, 0.01²) + 0.5·N(0.8, 0.05²), oříznutý na ℓ>0"""
    if np.random.random() < 0.5:
        ell = float(np.random.normal(0.1, 0.01))
    else:
        ell = float(np.random.normal(0.8, 0.05))
    ell = max(0.005, ell)
    random_hps = {
        "lengthscale": ell,
        "outputscale": 1.0,
        "noise":       0.01,
    }
    return get_batch_for_gp(batch_size, seq_len, num_features,
                             device=device, hyperparameters=random_hps, **kwargs)


def load_for_inference(checkpoint_path, device='cpu'):
    """Načte libovolný PFN checkpoint. Detekuje nlayers automaticky."""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f'Model nenalezen: {checkpoint_path}')

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = checkpoint.get('config', {})

    if 'num_features' in config:
        num_features     = config['num_features']
        max_dataset_size = config['max_dataset_size']
        criterion        = checkpoint['criterion']
        borders          = criterion.borders.tolist()
        nlayers          = config.get('nlayers', 6)
        get_batch_fn     = get_batch_for_gp
        prior_kwargs     = {'num_features': num_features, 'hyperparameters': config.get('hps', {})}
    elif 'priors' in config:
        num_features     = config['priors'][0]['prior_kwargs']['num_features']
        max_dataset_size = config['batch_shape_sampler']['max_seq_len']
        borders          = config['model']['criterion']['borders']
        nlayers          = config['model'].get('nlayers', 6)
        get_batch_fn     = get_batch_for_gp_random_hps
        prior_kwargs     = {'num_features': num_features, 'hyperparameters': {}}
        criterion        = None
    else:
        raise ValueError(f'Neznámý formát checkpointu. Klíče: {list(config.keys())}')

    state_dict = checkpoint.get('model_state_dict', {})
    layer_indices = [int(k.split('.')[2]) for k in state_dict
                     if k.startswith('transformer_layers.layers.')]
    if layer_indices:
        nlayers = max(layer_indices) + 1

    model_config = MainConfig(
        priors=[AdhocPriorConfig(get_batch_methods=[get_batch_fn], prior_kwargs=prior_kwargs)],
        optimizer=OptimizerConfig('adamw', lr=0.0003),
        model=TransformerConfig(
            criterion=BarDistributionConfig(full_support=True, borders=borders),
            emsize=512, nhead=8, nhid=1024, nlayers=nlayers,
            features_per_group=1, attention_between_features=False,
            encoder=EncoderConfig(constant_normalization_mean=0.5,
                                  constant_normalization_std=math.sqrt(1/12))
        ),
        batch_shape_sampler=BatchShapeSamplerConfig(
            batch_size=2, max_seq_len=max_dataset_size,
            min_num_features=num_features, max_num_features=num_features
        ),
        epochs=1, steps_per_epoch=1, num_workers=0,
    )

    dummy_result = train(model_config, device=device, reusable_config=False)
    model = dummy_result['model']
    model.load_state_dict(state_dict)
    if criterion is not None:
        model.criterion = criterion
    model.to(device)
    model.eval()

    epoch = checkpoint.get('epoch', '?')
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  nlayers={nlayers}, epoch={epoch}, {n_params:.2f}M params')
    return model, epoch


def reset_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available(): torch.mps.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

print("✓ load_for_inference + prior wrappery připraveny")

## Experiment 4: Hyperparametrická nejistota a prediktivní distribuce

### Q4 — Crossover jako funkce šíře prioru (Step 5 z PFN-GP2.md)

**Teorie:** PFN je meta-trénován přes prior $p(\theta)$ kde $\theta = (\ell, \sigma_f^2, \sigma_n^2)$.
Při malém $n$ je posterior $p(\theta | X, y)$ plochý — ML-II nemá dost dat pro spolehlivou identifikaci $\theta$.
PFN nese prior z meta-tréninku, takže jeho predikce jsou lepší.
Při velkém $n$ se ML-II přiblíží pravému $\theta$ a PFN výhoda zmizí.

**Crossover $n^*$:** hodnota $n$ kde NLL(PFN) = NLL(GP-ML). Závisí na šíři prioru:
- Úzký prior ($\ell \sim \text{LN}(-1, 0.2^2)$): ML-II identifikuje $\theta$ rychle → malé $n^*$
- Střední prior (existující modely, $\ell \sim U(0.05, 1.0)$): střední $n^*$
- Široký prior ($\ell \sim \text{LN}(-1, 1.5^2)$): ML-II potřebuje hodně bodů → velké $n^*$

**Test:** Srovnáme NLL(PFN) − NLL(GP-ML) jako funkci $n$ pro každý model.

---

### Q5 — Tvar prediktivní distribuce pod hyperparametrickou nejistotou (Step 6 z PFN-GP2.md)

**Teorie:** Marginalizace přes $\theta$ dává **směs Gaussiánů**:
$$p(y_* | x_*, X, y) = \int \mathcal{N}(\mu_\theta, \sigma^2_\theta)\, p(\theta | X, y)\, d\theta$$

Pokud je $p(\theta | X, y)$ bimodální, prediktivní distribuce může být **bimodální nebo heavy-tailed**.
`BarDistribution` tuto distribuci reprezentovat umí (na rozdíl od Gaussovské hlavy).

**Test:** Natrénujeme PFN na bimodálním prioru $\ell \sim 0.5\cdot N(0.1, 0.01^2) + 0.5\cdot N(0.8, 0.05^2)$.
Pro ambiguní kontext (data konzistentní s oběma $\ell$) vizualizujeme BarDistribution výstup
vs. oracle směs $0.5\cdot \mathcal{N}(\mu_{0.1}, \sigma^2_{0.1}) + 0.5\cdot \mathcal{N}(\mu_{0.8}, \sigma^2_{0.8})$.

In [ ]:
# =============================================
# TRÉNOVACÍ FUNKCE (přepoužitelná)
# =============================================

def train_pfn_with_prior(get_batch_fn, save_path, epochs=300,
                          num_features=1, max_dataset_size=128,
                          batch_size=64, steps_per_epoch=100, nlayers=6):
    """
    Natrénuje PFN model s danou prior funkcí.
    Ukládá checkpoint každých 50 epoch + finální.

    Parametry:
        get_batch_fn:    funkce vracející Batch (musí mít signaturu jako get_batch_for_gp)
        save_path:       cesta k výslednému checkpointu (.pth)
        epochs:          počet epoch
    """
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    save_every = 50

    def save_fn(checkpoint, path):
        ep = checkpoint['epoch']
        torch.save(checkpoint, path)
        if ep % save_every == 0:
            ep_path = path.replace('.pth', f'_epoch_{ep}.pth')
            torch.save(checkpoint, ep_path)
            print(f'  Checkpoint: {ep_path}')

    print(f'Vzorkuji BarDistribution borders ({get_batch_fn.__name__})...')
    ys = get_batch_fn(10000, max_dataset_size, num_features, device='cpu').target_y
    borders = bar_dist_module.get_bucket_borders(num_outputs=1000, ys=ys.cpu()).tolist()
    print(f'  Borders: y ∈ [{min(borders):.2f}, {max(borders):.2f}]')

    config = MainConfig(
        train_state_dict_save_path=save_path,
        train_state_dict_load_path=save_path,
        priors=[AdhocPriorConfig(
            get_batch_methods=[get_batch_fn],
            prior_kwargs={'num_features': num_features, 'hyperparameters': {}}
        )],
        optimizer=OptimizerConfig('adamw', lr=0.0003),
        model=TransformerConfig(
            criterion=BarDistributionConfig(full_support=True, borders=borders),
            emsize=512, nhead=8, nhid=1024, nlayers=nlayers,
            features_per_group=1, attention_between_features=False,
            encoder=EncoderConfig(constant_normalization_mean=0.5,
                                  constant_normalization_std=math.sqrt(1/12))
        ),
        batch_shape_sampler=BatchShapeSamplerConfig(
            batch_size=batch_size, max_seq_len=max_dataset_size,
            min_num_features=num_features, max_num_features=num_features
        ),
        epochs=epochs,
        warmup_epochs=epochs // 4,
        steps_per_epoch=steps_per_epoch,
        num_workers=0,
    )

    print(f'Začínám trénink: {nlayers} vrstev, {epochs} epoch, device={device}')
    result = train(config, device=device, reusable_config=False,
                   save_object_function=save_fn)
    print(f'✓ Trénink dokončen → {save_path}')
    return result

print("✓ train_pfn_with_prior připraven")

---
## TRÉNOVÁNÍ — spusť jen jednou (ideálně na GPU/Helios)

Buňky níže trénují tři nové modely:
- `pfn_narrow_prior_6layer.pth` — úzký prior ($\ell \sim \text{LN}(-1, 0.2^2)$) pro Q4
- `pfn_wide_prior_6layer.pth` — široký prior ($\ell \sim \text{LN}(-1, 1.5^2)$) pro Q4
- `pfn_bimodal_prior_6layer.pth` — bimodální prior ($\ell \sim 0.5 N(0.1,\cdot) + 0.5 N(0.8,\cdot)$) pro Q5

Pokud checkpointy již existují, buňky je přeskočí. **Trénink trvá cca 30–60 min na GPU.**

In [ ]:
# =============================================
# TRÉNOVÁNÍ: Úzký prior (Q4)
# =============================================

NARROW_PRIOR_PATH = os.path.join('models', 'pfn_narrow_prior_6layer.pth')

if os.path.exists(NARROW_PRIOR_PATH):
    print(f'✓ Úzký prior model již existuje: {NARROW_PRIOR_PATH}')
else:
    print('Trénuji model s úzkým priorem (ℓ ~ LN(-1, 0.2²))...')
    train_pfn_with_prior(
        get_batch_fn=get_batch_for_gp_narrow_prior,
        save_path=NARROW_PRIOR_PATH,
        epochs=300, nlayers=6
    )

In [ ]:
# =============================================
# TRÉNOVÁNÍ: Široký prior (Q4)
# =============================================

WIDE_PRIOR_PATH = os.path.join('models', 'pfn_wide_prior_6layer.pth')

if os.path.exists(WIDE_PRIOR_PATH):
    print(f'✓ Široký prior model již existuje: {WIDE_PRIOR_PATH}')
else:
    print('Trénuji model se širokým priorem (ℓ ~ LN(-1, 1.5²))...')
    train_pfn_with_prior(
        get_batch_fn=get_batch_for_gp_wide_prior,
        save_path=WIDE_PRIOR_PATH,
        epochs=300, nlayers=6
    )

In [ ]:
# =============================================
# TRÉNOVÁNÍ: Bimodální prior (Q5)
# =============================================

BIMODAL_PRIOR_PATH = os.path.join('models', 'pfn_bimodal_prior_6layer.pth')

if os.path.exists(BIMODAL_PRIOR_PATH):
    print(f'✓ Bimodální prior model již existuje: {BIMODAL_PRIOR_PATH}')
else:
    print('Trénuji model s bimodálním priorem (ℓ ~ 0.5·N(0.1) + 0.5·N(0.8))...')
    train_pfn_with_prior(
        get_batch_fn=get_batch_for_gp_bimodal_prior,
        save_path=BIMODAL_PRIOR_PATH,
        epochs=300, nlayers=6
    )

In [ ]:
# =============================================
# KONFIGURACE — načtení trénovaných modelů
# =============================================

MODEL_PATHS_Q4 = {
    'Úzký prior (σ_ℓ=0.2)':   os.path.join('models', 'pfn_narrow_prior_6layer.pth'),
    'Střední prior (default)': os.path.join('models', 'pfn_rand_hps_6layer.pth'),
    'Široký prior (σ_ℓ=1.5)':  os.path.join('models', 'pfn_wide_prior_6layer.pth'),
}

BIMODAL_MODEL_PATH = os.path.join('models', 'pfn_bimodal_prior_6layer.pth')

print('Načítám modely pro Q4...')
MODELS_Q4 = {}
for name, path in MODEL_PATHS_Q4.items():
    if os.path.exists(path):
        MODELS_Q4[name], _ = load_for_inference(path, device)
    else:
        print(f'  Nenalezeno (spusť trénink výše): {path}')

print('\nNačítám bimodální model pro Q5...')
model_bimodal = None
if os.path.exists(BIMODAL_MODEL_PATH):
    model_bimodal, _ = load_for_inference(BIMODAL_MODEL_PATH, device)
else:
    print(f'  Nenalezeno (spusť trénink výše): {BIMODAL_MODEL_PATH}')

reset_seed(42)
print(f'\nNačteno {len(MODELS_Q4)} modelů pro Q4')
print(f'Bimodální model: {"✓" if model_bimodal else "✗ chybí"}')

In [ ]:
# =============================================
# Q4: NLL CROSSOVER — funkce
# =============================================

def rbf_kernel(x1, x2, ls, scale=1.0):
    return scale * np.exp(-(x1[:, None] - x2[None, :]) ** 2 / (2 * ls**2))


def gp_exact_posterior(train_x, train_y, test_x, ls, noise, scale=1.0):
    """Přesný GP posterior mean a std."""
    K    = rbf_kernel(train_x, train_x, ls, scale)
    Kn   = K + noise * np.eye(len(train_x))
    Ks   = rbf_kernel(test_x, train_x, ls, scale)
    Kss  = rbf_kernel(test_x, test_x, ls, scale)
    try:
        L = np.linalg.cholesky(Kn)
        alpha = np.linalg.solve(L.T, np.linalg.solve(L, train_y))
        v = np.linalg.solve(L, Ks.T)
        mu  = Ks @ alpha
        var = np.diag(Kss) - np.sum(v**2, axis=0)
        std = np.sqrt(np.maximum(var, 1e-8))
    except np.linalg.LinAlgError:
        mu  = np.zeros(len(test_x))
        std = np.ones(len(test_x))
    return mu, std


def compute_gp_ml_nll(train_x_np, train_y_np, test_x_np, test_y_np):
    """
    Fituje GP s ML-II (sklearn) a spočítá prediktivní NLL na testovacích bodech.
    Optimalizuje ConstantKernel * RBF + WhiteKernel.
    """
    kernel = (ConstantKernel(1.0, (1e-3, 1e3))
              * RBF(0.3, (1e-3, 10.0))
              + WhiteKernel(0.01, (1e-5, 1.0)))
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, normalize_y=False)
    try:
        gp.fit(train_x_np.reshape(-1, 1), train_y_np)
        mu, sigma = gp.predict(test_x_np.reshape(-1, 1), return_std=True)
        sigma = np.maximum(sigma, 1e-6)
        nll = -float(np.mean(stats.norm.logpdf(test_y_np, mu, sigma)))
        return nll
    except Exception:
        return np.nan


def compute_pfn_nll(model, train_x, train_y, test_x, test_y, device):
    """
    Spočítá NLL z BarDistribution výstupu PFN.
    NLL(y) = -(log P(bin_k) - log width_k)   kde y leží v binu k.
    """
    model.eval()
    with torch.no_grad():
        logits = model(
            train_x.unsqueeze(0).to(device),
            train_y.unsqueeze(0).to(device),
            test_x.unsqueeze(0).to(device)
        )  # [1, n_test, n_bins]

        borders     = model.criterion.borders.cpu()
        bin_widths  = borders[1:] - borders[:-1]              # [n_bins]
        log_bin_w   = torch.log(bin_widths.clamp(min=1e-10))

        test_y_cpu  = test_y.cpu().squeeze(-1)                # [n_test]
        # Najdi bin pro každý testovací bod
        bin_idx = torch.searchsorted(borders[1:-1].contiguous(),
                                     test_y_cpu.contiguous())
        bin_idx = bin_idx.clamp(0, len(bin_widths) - 1)

        log_probs = F.log_softmax(logits[0], dim=-1)          # [n_test, n_bins]
        # NLL = -(log p_k - log w_k) = -(log (p_k/w_k)) = log density
        nll = -(log_probs.gather(-1, bin_idx.unsqueeze(-1)).squeeze(-1)
                - log_bin_w[bin_idx]).mean().item()

        return float(nll)


def run_crossover_experiment(models_dict, n_values, n_test_points,
                              n_instances, true_hps, device):
    """
    Pro každý model a každé n spočítá:
      NLL_PFN(n)   — prediktivní NLL PFN modelu
      NLL_GP_ML(n) — prediktivní NLL GP s ML-II odhadem θ

    Vrací: dict {model_name: {'n': list, 'pfn_nll': list, 'gp_nll': list, 'delta': list}}
    Kde delta = NLL_PFN - NLL_GP_ML (< 0 = PFN vítězí, > 0 = GP-ML vítězí)
    """
    results = {name: {'n': [], 'pfn_nll': [], 'pfn_se': [],
                      'gp_nll': [], 'gp_se': [], 'delta': []}
               for name in models_dict}

    ls     = true_hps['lengthscale']
    noise  = true_hps['noise']
    scale  = true_hps.get('outputscale', 1.0)

    for n in n_values:
        print(f'  n={n}: ', end='', flush=True)
        gp_nlls  = []
        pfn_nlls = {name: [] for name in models_dict}

        for _ in range(n_instances):
            # Generuj dataset s pravými HP
            batch = get_batch_for_gp(
                batch_size=1, seq_len=n + n_test_points, num_features=1,
                device='cpu', hyperparameters=true_hps
            )
            train_x = batch.x[0, :n]                # [n, 1]
            train_y = batch.y[0, :n]                 # [n]
            test_x  = batch.x[0, n:n+n_test_points] # [n_test, 1]
            test_y  = batch.y[0, n:n+n_test_points]  # [n_test]  (true GP samples)

            train_x_np = train_x.numpy().reshape(-1)
            train_y_np = train_y.numpy().reshape(-1)
            test_x_np  = test_x.numpy().reshape(-1)

            # GP přesný posterior jako "test_y" pro NLL (GP-ML predikuje distribuce)
            # Použijeme jako test_y skutečné GP samples z batch
            test_y_np = test_y.numpy().reshape(-1)

            # GP-ML NLL
            gp_nll = compute_gp_ml_nll(train_x_np, train_y_np, test_x_np, test_y_np)
            if not np.isnan(gp_nll) and abs(gp_nll) < 100:
                gp_nlls.append(gp_nll)

            # PFN NLL pro každý model
            for name, model in models_dict.items():
                try:
                    nll = compute_pfn_nll(model, train_x, train_y, test_x, test_y, device)
                    if not np.isnan(nll) and abs(nll) < 100:
                        pfn_nlls[name].append(nll)
                except Exception:
                    continue

        gp_mean = np.mean(gp_nlls) if gp_nlls else np.nan
        gp_se   = np.std(gp_nlls) / np.sqrt(max(len(gp_nlls), 1)) if gp_nlls else np.nan

        for name in models_dict:
            v = pfn_nlls[name]
            m = np.mean(v) if v else np.nan
            s = np.std(v) / np.sqrt(max(len(v), 1)) if v else np.nan
            results[name]['n'].append(n)
            results[name]['pfn_nll'].append(m)
            results[name]['pfn_se'].append(s)
            results[name]['gp_nll'].append(gp_mean)
            results[name]['gp_se'].append(gp_se)
            results[name]['delta'].append(m - gp_mean)
            print(f'{name}:Δ={m - gp_mean:.2f}  ', end='', flush=True)
        print()

    return results


def plot_crossover(results, title='Q4: NLL crossover vs. šíře prioru'):
    """Zobrazí delta NLL = NLL_PFN - NLL_GP_ML jako funkci n pro každý model."""
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = plt.cm.tab10.colors

    for i, (name, res) in enumerate(results.items()):
        ax.plot(res['n'], res['delta'], '-o', color=colors[i], lw=2, ms=7, label=name)
        ax.fill_between(res['n'],
                        np.array(res['delta']) - np.array(res['pfn_se']),
                        np.array(res['delta']) + np.array(res['pfn_se']),
                        alpha=0.15, color=colors[i])

    ax.axhline(0, color='black', lw=1.5, ls='--', label='NLL_PFN = NLL_GP-ML (crossover)')
    ax.set_xlabel('Počet tréninkových bodů n', fontsize=12)
    ax.set_ylabel('ΔNLL = NLL_PFN − NLL_GP-ML', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.set_xscale('log')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

print("✓ Q4 funkce připraveny")

In [ ]:
# =============================================
# SPUŠTĚNÍ Q4
# =============================================

if not MODELS_Q4:
    print("Modely pro Q4 nejsou k dispozici. Spusť tréninkové buňky výše.")
else:
    TRUE_HPS = {'lengthscale': 0.3, 'outputscale': 1.0, 'noise': 0.01}
    N_VALUES     = [2, 5, 10, 20, 50, 100]
    N_TEST_Q4    = 10
    N_INSTANCES  = 50    # instance per n (GP-ML je pomalý)

    reset_seed(42)
    print('Spouštím Q4: NLL crossover experiment...')
    q4_results = run_crossover_experiment(
        MODELS_Q4, N_VALUES, N_TEST_Q4, N_INSTANCES, TRUE_HPS, device
    )
    plot_crossover(q4_results)

    # Odhad n* pro každý model (kde delta přechází přes 0)
    print('\n=== Odhadované crossover n* ===')
    for name, res in q4_results.items():
        deltas = np.array(res['delta'])
        ns     = np.array(res['n'])
        # Najdi první n kde delta > 0 (GP-ML začíná vítězit)
        pos = np.where(deltas > 0)[0]
        if len(pos) > 0:
            n_star = ns[pos[0]]
            print(f'  {name}: n* ≈ {n_star}')
        else:
            print(f'  {name}: crossover nenalezen (PFN vítězí i při n={ns[-1]})')

## Interpretace Q4

**Osa X (log):** počet tréninkových bodů $n$  
**Osa Y:** $\Delta\text{NLL} = \text{NLL}_{\text{PFN}} - \text{NLL}_{\text{GP-ML}}$

- $\Delta < 0$: PFN má lepší predikce (nižší NLL = vyšší pravděpodobnost) → PFN vítězí
- $\Delta > 0$: GP-ML vítězí
- $\Delta = 0$: crossover bod $n^*$

### Jak číst grafy

| Pozorování | Závěr |
|---|---|
| Úzký prior: malé $n^*$ | ML-II rychle identifikuje $\ell$ → PFN výhoda rychle mizí |
| Široký prior: velké $n^*$ | ML-II potřebuje hodně dat → PFN nese užitečný prior déle |
| $n^* \propto \sigma_\ell$ | Čistý predikát z teorie potvrzen |
| Všechny modely mají stejné $n^*$ | Prior šíře neovlivňuje crossover |

### Teoretický základ

Výhoda PFN pochází z druhého členu rozptylu směsi Gaussiánů:
$$\text{Var}[y_*] = \underbrace{\mathbb{E}_\theta[\sigma^2_\theta]}_{\text{noise}} + \underbrace{\text{Var}_\theta[\mu_\theta]}_{\text{nejistota o }\theta}$$

Druhý člen klesá jak $p(\theta | X, y)$ se koncentruje (při rostoucím $n$).
Úzký prior → $p(\theta)$ je úzký → $p(\theta | X, y)$ se koncentruje rychle (i při malém $n$) → malé $n^*$.
Široký prior → opak.

In [ ]:
# =============================================
# Q5: TVAR BARDISTRIBUTION POD BIMODÁLNÍM PRIOREM — funkce
# =============================================

def gp_posterior_pdf(y_vals, train_x_np, train_y_np, test_x_val, ls, noise, scale=1.0):
    """Hustota pravděpodobnosti GP posterioru N(mu, sigma²) v hodnotách y_vals."""
    mu, sigma = gp_exact_posterior(
        train_x_np, train_y_np, np.array([test_x_val]), ls, noise, scale
    )
    return stats.norm.pdf(y_vals, loc=mu[0], scale=sigma[0]), mu[0], sigma[0]


def get_pfn_bar_distribution(model, train_x, train_y, test_x, test_idx=0, device='cpu'):
    """
    Extrahuje BarDistribution výstup PFN pro jeden testovací bod.
    Vrací: (bin_centers, density, borders)
    kde density[i] = P(y ∈ bin_i) / width_i  (hustota, ne pravděpodobnost)
    """
    model.eval()
    with torch.no_grad():
        logits = model(
            train_x.unsqueeze(0).to(device),
            train_y.unsqueeze(0).to(device),
            test_x.unsqueeze(0).to(device)
        )
        probs   = torch.softmax(logits[0, test_idx], dim=-1).cpu().numpy()
        borders = model.criterion.borders.cpu().numpy()

    bin_widths  = borders[1:] - borders[:-1]
    bin_centers = (borders[:-1] + borders[1:]) / 2
    density     = probs / np.maximum(bin_widths, 1e-10)
    return bin_centers, density, borders, probs


def create_ambiguous_context(n_context=5, x_range=(0, 3), test_x_val=1.5,
                              ls=0.8, noise=0.01, seed=0):
    """
    Vytvoří kontext konzistentní s oběma ℓ=0.1 a ℓ=0.8:
    - Málo bodů (n=5) → posterior over θ je stále plochý
    - Rovnoměrně rozmístěné X → jsou vzdálenosti nevýrazné
    """
    rng = np.random.RandomState(seed)
    train_x = np.linspace(x_range[0], x_range[1], n_context)
    # Sample y ze širokého GP (ls=0.8 → hladká funkce)
    hps = {'lengthscale': ls, 'outputscale': 1.0, 'noise': noise}
    batch = get_batch_for_gp(
        batch_size=1, seq_len=n_context + 1, num_features=1,
        device='cpu', hyperparameters=hps
    )
    train_x_t = batch.x[0, :n_context]
    train_y_t = batch.y[0, :n_context]
    test_x_t  = torch.tensor([[test_x_val]], dtype=torch.float32)
    return train_x_t, train_y_t, test_x_t


def visualize_bardistribution_vs_gp(model_bimodal, train_x, train_y, test_x,
                                     test_x_val, device='cpu', title=''):
    """
    Vizualizuje BarDistribution výstup PFN vs. GP posterior pro oba ℓ a oracle směs.

    Zobrazí:
    - PFN BarDistribution (histogram)
    - GP posterior N(μ_{0.1}, σ_{0.1}²) — modrá přerušovaná
    - GP posterior N(μ_{0.8}, σ_{0.8}²) — zelená přerušovaná
    - Oracle směs 0.5·N_{0.1} + 0.5·N_{0.8} — červená
    """
    train_x_np = train_x.numpy().reshape(-1)
    train_y_np = train_y.numpy().reshape(-1)

    # PFN výstup
    bin_centers, density, borders, probs = get_pfn_bar_distribution(
        model_bimodal, train_x, train_y, test_x, test_idx=0, device=device
    )

    # Omezte osu Y na smysluplný rozsah
    y_min = np.percentile(train_y_np, 2) - 2
    y_max = np.percentile(train_y_np, 98) + 2
    y_vals = np.linspace(y_min, y_max, 300)

    # GP posteriorní hustoty
    noise = 0.01
    pdf_01, mu_01, s_01 = gp_posterior_pdf(y_vals, train_x_np, train_y_np, test_x_val,
                                            ls=0.1, noise=noise)
    pdf_08, mu_08, s_08 = gp_posterior_pdf(y_vals, train_x_np, train_y_np, test_x_val,
                                            ls=0.8, noise=noise)
    oracle_pdf = 0.5 * pdf_01 + 0.5 * pdf_08

    # Maska na y_min..y_max pro PFN hustotu
    mask = (bin_centers >= y_min) & (bin_centers <= y_max)

    fig, ax = plt.subplots(figsize=(10, 6))

    # PFN hustota
    bin_w = borders[1:] - borders[:-1]
    ax.bar(bin_centers[mask], density[mask], width=bin_w[mask],
           alpha=0.55, color='steelblue', label='PFN BarDistribution')

    # GP posterior hustoty
    ax.plot(y_vals, pdf_01, 'b--', lw=2, label=f'GP ℓ=0.1 (μ={mu_01:.2f}, σ={s_01:.2f})')
    ax.plot(y_vals, pdf_08, 'g--', lw=2, label=f'GP ℓ=0.8 (μ={mu_08:.2f}, σ={s_08:.2f})')
    ax.plot(y_vals, oracle_pdf, 'r-', lw=2.5, label='Oracle směs 0.5·GP₀.₁ + 0.5·GP₀.₈')

    ax.set_xlim(y_min, y_max)
    ax.set_xlabel('y*', fontsize=12)
    ax.set_ylabel('Hustota pravděpodobnosti', fontsize=12)
    ax.set_title(title or f'Q5: PFN prediktivní distribuce (x*={test_x_val:.1f})', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Výpis statistik
    pfn_mean = float(np.sum(probs * bin_centers))
    pfn_var  = float(np.sum(probs * (bin_centers - pfn_mean)**2))
    oracle_mean = 0.5 * mu_01 + 0.5 * mu_08
    print(f'  PFN:         mean={pfn_mean:.3f}, std={np.sqrt(pfn_var):.3f}')
    print(f'  GP ℓ=0.1:    mean={mu_01:.3f}, std={s_01:.3f}')
    print(f'  GP ℓ=0.8:    mean={mu_08:.3f}, std={s_08:.3f}')
    print(f'  Oracle směs: mean={oracle_mean:.3f}')

print("✓ Q5 funkce připraveny")

In [ ]:
# =============================================
# SPUŠTĚNÍ Q5
# =============================================

if model_bimodal is None:
    print("Bimodální model není k dispozici. Spusť tréninkovou buňku výše.")
else:
    reset_seed(0)

    TEST_X_VAL = 1.5   # testovací bod
    N_CONTEXT  = 5     # málo bodů → prior dominuje → ambiguita

    # --- Případ 1: Ambiguní kontext (data konzistentní s oběma ℓ) ---
    print('=== Případ 1: Ambiguní kontext (n=5, ℓ_gen=0.8, body rovnoměrně rozmístěny) ===')
    train_x_amb, train_y_amb, test_x_t = create_ambiguous_context(
        n_context=N_CONTEXT, x_range=(0, 3), test_x_val=TEST_X_VAL,
        ls=0.8, noise=0.01, seed=42
    )
    visualize_bardistribution_vs_gp(
        model_bimodal, train_x_amb, train_y_amb, test_x_t,
        TEST_X_VAL, device=device,
        title='Q5: Ambiguní kontext — PFN by měl být bimodální nebo heavy-tailed'
    )

    # --- Případ 2: Jednoznačný kontext (hustě vzorkovaná data z ℓ=0.8) ---
    print('\n=== Případ 2: Jednoznačný kontext (n=20, ℓ_gen=0.8, husté vzorkování) ===')
    hps_clear = {'lengthscale': 0.8, 'outputscale': 1.0, 'noise': 0.01}
    batch_clear = get_batch_for_gp(
        batch_size=1, seq_len=21, num_features=1,
        device='cpu', hyperparameters=hps_clear
    )
    train_x_clear = batch_clear.x[0, :20]
    train_y_clear = batch_clear.y[0, :20]
    test_x_clear  = torch.tensor([[TEST_X_VAL]], dtype=torch.float32)

    visualize_bardistribution_vs_gp(
        model_bimodal, train_x_clear, train_y_clear, test_x_clear,
        TEST_X_VAL, device=device,
        title='Q5: Jednoznačný kontext — PFN by se měl soustředit na ℓ=0.8'
    )

## Interpretace Q5

### Co hledáme

**Ambiguní kontext** (málo bodů, rovnoměrné rozmístění $X$):
- Data jsou konzistentní jak s $\ell = 0.1$ (malé wiggly oscilace), tak s $\ell = 0.8$ (hladká funkce)
- $p(\theta | X, y)$ by měla být **bimodální** nebo plochá
- Proto by prediktivní distribuce měla být **heavy-tailed nebo bimodální**

**Jednoznačný kontext** (n=20, hustě vzorkovaná data z $\ell = 0.8$):
- Hustota vzorkování určuje, jakou frekvenci funkce „vidíme"
- $p(\theta | X, y)$ by se měla koncentrovat kolem $\ell \approx 0.8$
- PFN output by se měl blížit $\mathcal{N}(\mu_{0.8}, \sigma^2_{0.8})$

### Jak číst grafy

| Pozorování | Závěr |
|---|---|
| PFN ambiguní: těžká ramena nebo dva kopce | Model zachytil hyperparametrickou nejistotu ✓ |
| PFN ambiguní: úzká Gaussiána | Model se zavázal k jednomu $\ell$ — ztráta nejistoty ✗ |
| PFN jednoznačný: blízko GP $\ell=0.8$ | Model identifikoval $\ell$ z kontextu ✓ |
| PFN jednoznačný: stále bimodální | Model nereaguje na kontext ✗ |

### Proč BarDistribution

Standardní výstupní hlava (Gaussiánská) nemůže reprezentovat bimodální distribuce.
`BarDistribution` diskretizuje výstup do 1000 košů a může se přiblížit **libovolné** hustotě,
včetně směsí. Pokud trénink fungoval, model by měl využít tuto kapacitu právě pro ambiguní kontexty.

Srovnání s **oracle směsí** $0.5 \mathcal{N}_{0.1} + 0.5 \mathcal{N}_{0.8}$ ukáže, jak blízko je PFN
k teoreticky optimálnímu chování Bayesovské marginalizace.